# Agentic Workflow

Aentic RAG3 게시물 까지는 Langchain을 이용하여 Agent를 만들었다면, 지금부터는 Langchain의 자매격인 **Langgraph** 프레임워크를 이용해서 Agentic Workflow를 구성해보자!

### LangGraph 상세 가이드: Agentic Workflow 구축을 위한 핵심 도구

LangGraph는 LangChain Expression Language (LCEL)를 기반으로 구축되었으며, LLM 기반 Agent가 단순한 선형 체인(Chain)을 넘어 **복잡한 의사 결정 및 반복(Loop)** 을 수행할 수 있도록 돕습니다.

#### 1. LangGraph의 핵심 개념

1.1. 상태 (State) - 랭그래프 워크플로의 핵심이다.  

>모든 LangGraph 워크플로우의 핵심입니다. 상태는 노드(Node) 간에 전달되는 **데이터 컨테이너** 입니다. Agent의 입력, 출력, 검색 결과, 도구 사용 기록 등 모든 정보가 이 상태에 저장됩니다.  
>구현: TypedDict를 사용하여 상태의 스키마를 정의하며, Annotated[List[Any], operator.add]와 같은 구문을 사용하여 리스트를 병합(Aggregate)하는 방법을 지정합니다.(구조화된 스키마 안에 있는 내용을 바탕으로 state들이 각 노드를 거치면서 state에 저장을 한다)

1.2. 노드 (Nodes)

>워크플로우 내에서 **특정 작업을 수행하는 함수 또는 컴포넌트** 입니다. 즉, 노드는 내가 작업하고 싶은 task를 정의한 것이다. 예를 들어 전처리 하는 노드, llm과 체이닝하는 노드, 답변을 얻는 노드 등 이렇게 구성한다. 

>종류:

>>Agent 노드: LLM을 호출하여 추론(Thought)하고 다음 행동(Action)을 결정합니다.
>>Tool 노드: Agent가 요청한 도구를 실행하고 결과를 상태에 반영합니다.
>>Custom 노드: 데이터 전처리, 로그 기록, 조건 평가 등 사용자 정의 로직을 수행합니다.

1.3. 엣지 (Edges)

>노드 간의 연결을 정의하며, 데이터가 흐르는 방향을 지정합니다.  
>기본 엣지 (add_edge): 한 노드에서 다음 노드로 무조건 이동합니다 (예: tools -> agent).  
>시작 지점 (set_entry_point): 워크플로우가 시작되는 최초 노드를 지정합니다.  
>종료 지점 (END): 워크플로우가 끝나는 지점을 지정합니다.  

1.4. 조건부 엣지 (Conditional Edges) - a에서 상황에 따라 b c d로 가거나 , 아니면 노드로 가지 않을 수도 있다. 그래서 워크플로우의 동적 흐름을 담당한다. 보통 라우팅 함수를 사용한다. 

>가장 중요한 기능으로, 워크플로우의 동적 흐름을 담당합니다.  
>구현: 노드의 출력을 받아 다음 노드의 이름을 반환하는 **라우터 함수(Router Function)**를 사용합니다. (예: LLM이 도구 사용을 결정했으면 'tools', 최종 답변이면 'end'를 반환).


랭그래프의 핵심은 바로 '흐름을 제어하는 것'이다. 여기서 엣지는 직진만 가능한 일반통행 도로이고, 조건부 엣지는 내비게이션이 안내해주는 갈림길이다!!  기본 엣지(add_edge)는 고민할 필요 없이 A노드 작업 끝나면 무조건 B노드 작업으로 간다. 하지만, 조건부 엣지(add_conditional_edges)는 A 노드의 결과물인 State를 보고 그 다음으로 어디를 갈지 그때그때 결정하는 분기점이다. 이때 길을 안내해주는 역할이 '라우터 함수'이다.



#### 2. Agentic Workflow에서의 LangGraph 역할

- Agentic RAG에서 LangGraph는 다음과 같은 복잡한 패턴을 효율적으로 구현합니다.
- ReAct 루프 구현: Agent 노드 $\leftrightarrow$ Tool 노드 사이를 반복적으로 순환하여, LLM이 추론하고 도구를 사용하며 문제를 단계적으로 해결합니다.(Agent 노드는 두뇌에 해당하는 노드, Tool 노드는 손과 발에 해당하는 노드 )
- Adaptive 라우팅: 질문 유형(내부 지식 vs. 외부 검색)이나 검색 결과 품질(CRAG)에 따라 워크플로우의 경로를 동적으로 변경합니다.
- 상태 지속성: 복잡한 반복 과정 중에도 모든 중간 단계(intermediate_steps)와 데이터를 AgentState를 통해 안전하게 유지하고 다음 추론에 활용합니다.


>> Agent 노드가 현재 State를 읽고 추론을 한다. -> 이를 바탕으로 Tool 호출 명령을 내린다. -> 이를 Tool 노드가 실행하고, 실행해서 얻은 실제 데이터를 State에 저장해서 다시 Agent에게 넘기고! 이런 식이다. 

**결국 LangGraph를 이용하면, agentic Rag에서 LangCahin만 사용하는 것 보다 더 커스텀 가능하고 복잡한 패턴의 구현이 가능하다. 또한 제어도 쉽고 보다 직관적이다.** 

# LangGraph 워크플로우 구현 단계 정리

LangGraph를 사용해 사용자의 질문을 LLM에 전달하고, 응답을 다시 상태에 누적하는 가장 기본적인 Agentic Workflow를 구현한다. 전체 구조는 `START -> chatbot -> END`로 이어지는 단일 노드 그래프이며, 메시지 상태를 중심으로 워크플로우가 실행된다.

## 1. LLM과 기본 라이브러리 준비

- `ChatGoogleGenerativeAI`를 사용해 Gemini 기반 LLM을 정의한다.
- 모델은 `gemini-2.5-flash-lite`, `temperature=0`으로 설정한다.
- `temperature=0`은 같은 질문에 대해 비교적 일관된 답변을 얻기 위한 설정이다.
- LangGraph 구성을 위해 `StateGraph`, `START`, `END`, `add_messages`를 import한다.
- 상태 타입 정의를 위해 `TypedDict`, `Annotated`를 사용한다.

## 2. State 정의하기

- LangGraph에서 `State`는 그래프 내부를 이동하는 데이터의 구조를 의미한다.
- 이 노트북에서는 `State`를 `TypedDict`로 정의하고, 내부에 `messages` 필드를 둔다.
- `messages: Annotated[list, add_messages]`는 메시지를 리스트로 관리하되, 새 메시지가 들어올 때 기존 메시지를 덮어쓰지 않고 누적하라는 의미이다.
- 즉, 각 노드가 반환하는 메시지는 `add_messages` 규칙에 따라 기존 대화 기록 뒤에 추가된다.
- 이 구조 덕분에 워크플로우 실행 중 사용자 입력과 LLM 응답을 하나의 메시지 히스토리로 관리할 수 있다.

## 3. Node 정의하기

- LangGraph에서 Node는 실제 작업을 수행하는 함수 또는 실행 단위이다.
- 이 노트북에서는 `chatbot(state: State)` 함수를 하나의 노드로 사용한다.
- `chatbot` 함수는 현재 상태에서 `state["messages"]`를 꺼낸다.
- 꺼낸 메시지를 `llm.invoke(messages)`에 전달해 LLM 응답을 생성한다.
- 반환값은 `{"messages": [llm.invoke(messages)]}` 형태이며, 이 응답은 State의 `messages`에 누적된다.
- 따라서 `chatbot` 노드는 사용자 메시지를 받아 LLM 답변을 생성하고, 그 결과를 다시 그래프 상태에 저장하는 역할을 한다.

## 4. Graph 생성 및 Node 추가하기

- `graph_builder = StateGraph(State)`로 그래프 빌더를 생성한다.
- 이 선언은 그래프 안에서 이동하는 데이터가 앞에서 정의한 `State` 구조를 따른다는 의미이다.
- `graph_builder.add_node("chatbot", chatbot)`으로 `chatbot` 함수를 그래프의 노드로 등록한다.
- 이때 `"chatbot"`은 그래프 내부에서 사용할 노드 이름이고, `chatbot`은 실제 실행될 함수이다.

## 5. Edge 정의하기

- Edge는 그래프에서 노드 사이의 실행 순서를 정의한다.
- `graph_builder.add_edge(START, "chatbot")`은 그래프가 시작되면 가장 먼저 `chatbot` 노드를 실행하라는 뜻이다.
- `graph_builder.add_edge("chatbot", END)`는 `chatbot` 노드 실행이 끝나면 그래프를 종료하라는 뜻이다.
- 현재 워크플로우는 조건 분기나 반복 없이 한 번의 LLM 호출 후 종료되는 단순한 구조이다.

## 6. Graph 컴파일하기

- `graph = graph_builder.compile()`을 실행해 정의한 State, Node, Edge를 실제 실행 가능한 그래프로 변환한다.
- 컴파일 전에는 그래프의 설계도에 가깝고, 컴파일 후에는 `invoke()`나 `stream()`으로 실행할 수 있는 객체가 된다.
- 이 단계가 끝나면 LangGraph 워크플로우를 실행할 준비가 완료된다.

## 7. Graph 실행하기

- 사용자 질문을 `question` 변수에 저장한다.
- `graph.stream({"messages": [("user", question)]})` 형태로 초기 State를 전달한다.
- 초기 State의 `messages`에는 사용자 메시지가 들어간다.
- `graph.stream()`은 그래프 실행 과정에서 발생하는 이벤트를 순차적으로 반환한다.
- 이 노트북에서는 각 이벤트의 `messages` 마지막 값을 꺼내 `value["messages"][-1].content`로 LLM의 최종 답변 내용을 출력한다.
- `graph.invoke()`가 최종 결과만 받는 방식이라면, `graph.stream()`은 실행 흐름을 실시간으로 확인할 수 있는 방식이다.

## 8. Graph 시각화하기

- `visualize_graph(graph)`를 사용해 컴파일된 그래프 구조를 시각화한다.
- `graph.get_graph().draw_mermaid()`를 사용하면 Mermaid 문법으로 그래프 구조를 출력할 수 있다.
- 출력된 Mermaid 코드는 `mermaid.live` 같은 도구에서 시각적으로 확인할 수 있다.
- 현재 그래프는 `START -> chatbot -> END` 형태로 표현된다.

## 전체 실행 흐름

1. 사용자 질문을 `messages`에 담아 그래프에 입력한다.
2. 그래프는 `START`에서 시작한다.
3. `START`는 `chatbot` 노드로 연결된다.
4. `chatbot` 노드는 현재 메시지 목록을 LLM에 전달한다.
5. LLM이 답변을 생성한다.
6. 생성된 답변이 `messages` 상태에 추가된다.
7. `chatbot` 노드 실행 후 `END`로 이동해 워크플로우가 종료된다.
8. `graph.stream()`을 통해 실행 이벤트와 응답을 확인한다.



## LangGraph 사용법

In [ ]:
## LangGraph chatbot

import os
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", temperature=0)

In [ ]:
# import nest_asyncio
# nest_asyncio.apply()

from typing import Annotated, TypedDict, Optional, List
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel, Field

In [ ]:
###### STEP 1. 상태(State) 정의 ######
class State(TypedDict): # TypedDict와 Annotated는 타입힌트 상세하게 달아주는 도구
    # 메시지 정의(list type 이며 add_messages 함수를 사용하여 메시지를 추가)
    messages: Annotated[list, add_messages] 
    
    # Annotated는 단순 타입 지정을 넘어서 부가 정보를 넣을게! 랭그래프 엔진은이 annotated 안에 적힌 부가정보를 읽어서 기능을 수행한다. 

    #docs : List[str] = Field("리트리벌을 통해 나온 도큐먼트", default=[])

# class State(BaseModel):
#     messages: List[str] = Field(..., description="대화 메시지들") 이런 방식으로 스키마 정의하기도 가능하지만, TypeDict 일반적으로 많이 쓴다. 

# 아하! 개발자가 만든 State는 딕셔너리구나. 그 안에 messages라는 리스트가 있네? 그런데 옆에 add_messages라는 특수 명령어가 적혀있으니까, 앞으로 노드들이 messages에 뭔가를 넘겨줄 때는 기존 꺼 덮어쓰지 말고 add_messages 함수를 실행해서 차곡차곡 쌓아줘야겠다!

In [ ]:
###### STEP 2. 노드(Node) 정의 ######
# LLM 정의
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", temperature=0)

# Node : 어떤 행위, Task 
# 챗봇 함수 정의
def chatbot(state: State):
    # 메시지 호출 및 반환
    messages = state["messages"]
    return {"messages": [llm.invoke(messages)]}

In [ ]:
###### STEP 3. 그래프(Graph) 정의, 노드 추가 ######

# 그래프 생성
graph_builder = StateGraph(State) 
# state를 인자로 받아서 state그래프 함수를 통해서 그래프를 생성한다. 
# 앞으로 이 그래프 안에서 돌아다닐 데이터는 이 State 양식을 따를거야! 라고 선언

# 노드 이름, 함수 혹은 callable 객체를 인자로 받아 노드를 추가
graph_builder.add_node("chatbot", chatbot) 
# 생성한 그래프의 메서드를 통해 우리가 만든 챗봇 함수를 넣어준다. 이때 각각의 노드는 state를 상속받게 된다. 

그래프를 정의한다는 것은 데이터가 어떤 순서와 규칙으로 움직일지 파이프라인을 조립하는 것을 의미한다. 노드를 정의하고 그냥 두는게 아니라, 그래프를 정의해서 이 분산된 요소들을 묶어 하나의 에이전트로 만드는 작업이 필요하다. 

시작점 지정: 사용자가 질문을 던졌을 때, 가장 먼저 실행할 노드는 무엇인가?

길 배치 (엣지): A 노드가 끝나면 B 노드로 가게 할 것인가, 아니면 조건에 따라 길을 나누어 줄 것인가?

도착점 지정: 어떤 조건이 만족되어야 이 전체 워크플로우를 종료(END)하고 사용자에게 답을 보여줄 것인가?

In [ ]:
###### STEP 4. 그래프 엣지(Edge) 추가 ######
# 시작 노드에서 챗봇 노드로의 엣지 추가
graph_builder.add_edge(START, "chatbot")

# 그래프에 엣지 추가
graph_builder.add_edge("chatbot", END)

In [ ]:
###### STEP 5. 그래프 컴파일(compile) ######
# 그래프 컴파일을 통해 하나의 '그래프'를 만들어준다. 
graph = graph_builder.compile()

In [ ]:
###### STEP 6. 그래프 실행 ######
question = "서울의 유명한 관광지 TOP 5와 각각의 소개에 대해 설명해줘"

# 그래프 이벤트 스트리밍
for event in graph.stream({"messages": [("user", question)]}):
    # 이벤트 값 출력
    for value in event.values():
        print(value["messages"][-1].content)
# 랭그래프에서 graph.stream()을 사용해 이벤트를 스트리밍하는 것은, 전체 워크플로우가 돌아가는 과정을 "실시간 생중계(모니터링)"하는 역할을 한다. 스트리밍을 쓰지 않고, 일반적으로 graph.invoke()를 썼다면 결과만 받는다. 
# 이벤트 스트리밍을 통해 중간 결과를 실시간으로 받을 수 있다. 

여기서 Chatbot이 대답하는 노드 이외에 그래프에서 또 다른 분기를 치고 싶다면, 그것도 따로 정의해 주면 되는 방식이다. 

In [ ]:
from utils import visualize_graph

visualize_graph(graph)
# 이 코드를 실행하면 그래프 형태의 구조를 시각화 해서 볼 수 있다. 

# https://mermaid.live/
print(graph.get_graph().draw_mermaid()) # 웹상에서 그래프 구조가 나온다. 이 그래프가 어떤 구조인지 시각화해서 볼 수 있다. 

## 요약

이 노트북의 LangGraph 워크플로우는 `State`, `Node`, `Edge`, `Compile`, `Run`의 흐름으로 구성된다. `State`는 메시지 데이터를 관리하고, `chatbot` 노드는 LLM을 호출해 답변을 생성하며, Edge는 `START -> chatbot -> END` 실행 순서를 정의한다. 이후 `compile()`로 그래프를 실행 가능한 객체로 만들고, `graph.stream()`으로 사용자 질문에 대한 응답을 이벤트 단위로 확인한다. 현재 구현은 단일 챗봇 노드만 가진 기본 구조지만, 여기에 검색 노드, 조건 분기, 반복 루프, 검증 노드 등을 추가하면 더 복잡한 Agentic RAG 또는 Agentic Workflow로 확장할 수 있다.